# Log Analysis

This notebook parses TensorBoard event logs under `tensorlogs/` and evaluation results under `logs/` to visualize training curves and summarize test outcomes.

_Run the cells below after launching the notebook from the repository root so that relative paths resolve correctly._


In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from tensorboard.backend.event_processing import event_accumulator

plt.rcParams["figure.figsize"] = (8, 4)


In [2]:
repo_root = Path.cwd().parent
tensorlogs_dir = repo_root / "tensorlogs"
logs_dir = repo_root / "logs"

print(f"TensorBoard runs: {sorted(p.name for p in tensorlogs_dir.glob('*') if p.is_dir())}")
print(f"Evaluation runs: {sorted(p.name for p in logs_dir.glob('*') if p.is_dir())}")


TensorBoard runs: ['2025-11-02_22-57-28_BC_PP', '2025-11-03_21-47-30_BC_PP', '2025-11-03_23-04-13_BC_PP']
Evaluation runs: ['2025-07-20-17-52-02-train-a2', '2025-11-02-22-45-40-test-pp-a2', '2025-11-02-22-47-09-test-grasp-a2', '2025-11-02-22-57-28-train-a2', '2025-11-02-23-07-20-test-grasp-a2-unseen', '2025-11-02-23-13-58-test-pp-a2-unseen', '2025-11-03-16-20-54-test-grasp-a2', '2025-11-03-16-44-04-test-grasp-a2-unseen', '2025-11-03-18-20-06-test-grasp-a2', '2025-11-03-18-42-16-test-grasp-a2', '2025-11-03-18-43-54-test-grasp-a2-unseen', '2025-11-03-19-08-00-test-grasp-a2-unseen', '2025-11-03-21-47-30-train-a2', '2025-11-03-23-04-59-test-grasp-a2', '2025-11-03-23-26-08-test-grasp-a2-unseen']


## TensorBoard Scalars

The helper below loads scalar histories from each event file under `tensorlogs/` into a single DataFrame.


In [3]:
def load_tensorboard_scalars(tensorlogs_dir):
    records = []
    if not tensorlogs_dir.exists():
        print(f"Directory not found: {tensorlogs_dir}")
        return pd.DataFrame(records)
    for run_path in sorted(tensorlogs_dir.iterdir()):
        if not run_path.is_dir():
            continue
        event_files = sorted(run_path.glob("events.out.tfevents.*"))
        if not event_files:
            print(f"No event files in {run_path.name}")
            continue
        for event_path in event_files:
            try:
                ea = event_accumulator.EventAccumulator(str(event_path), size_guidance={"scalars": 0})
                ea.Reload()
            except Exception as exc:
                print(f"Could not load {event_path.name}: {exc}")
                continue
            for tag in ea.Tags().get("scalars", []):
                for event in ea.Scalars(tag):
                    records.append({
                        "run": run_path.name,
                        "event_file": event_path.name,
                        "tag": tag,
                        "step": event.step,
                        "wall_time": event.wall_time,
                        "value": event.value,
                    })
    df = pd.DataFrame.from_records(records)
    if df.empty:
        return df
    return df.sort_values(["tag", "run", "step"]).reset_index(drop=True)


In [4]:
tb_scalars = load_tensorboard_scalars(tensorlogs_dir)
tb_scalars.head()


,run,event_file,tag,step,wall_time,value
0,2025-11-02_22-57-28_BC_PP,events.out.tfevents.1762120648.trossen-1,loss/epoch,0,1.762121e+09,3.155708
1,2025-11-02_22-57-28_BC_PP,events.out.tfevents.1762120648.trossen-1,loss/epoch,1,1.762121e+09,3.082009
2,2025-11-02_22-57-28_BC_PP,events.out.tfevents.1762120648.trossen-1,loss/epoch,2,1.762121e+09,3.029096
3,2025-11-02_22-57-28_BC_PP,events.out.tfevents.1762120648.trossen-1,loss/epoch,3,1.762121e+09,2.852364
4,2025-11-02_22-57-28_BC_PP,events.out.tfevents.1762120648.trossen-1,loss/epoch,4,1.762121e+09,2.723838


In [5]:
tb_scalars['tag'].unique()

array(['loss/epoch', 'loss/iteration'], dtype=object)

In [ ]:
sns.lineplot(
    tb

## Test Log Summaries

The utilities below read per-case result files written during evaluation runs under `logs/`.


In [3]:
METRIC_LABELS = {
    4: ["avg_success", "avg_step", "avg_success_step", "avg_reward"],
    5: ["avg_success", "avg_grasp_success", "avg_place_success", "avg_step", "avg_success_step"],
}

def detect_run_type(run_name):
    lowered = run_name.lower()
    if "train" in lowered:
        return "train"
    if "grasp" in lowered:
        return "grasp"
    if "pp" in lowered:
        return "pick-place"
    if "place" in lowered:
        return "place"
    return "unknown"

def parse_case_result(case_path):
    text = case_path.read_text().strip()
    if not text:
        return None
    tokens = text.split()
    float_positions = []
    float_values = []
    for idx, token in enumerate(tokens):
        try:
            value = float(token)
        except ValueError:
            continue
        float_positions.append(idx)
        float_values.append(value)
    if not float_values:
        return None
    first_value_pos = float_positions[0]
    language_tokens = tokens[:first_value_pos]
    language_goal = " ".join(language_tokens)
    return language_goal, float_values

def load_test_results(logs_dir):
    rows = []
    if not logs_dir.exists():
        print(f"Directory not found: {logs_dir}")
        return pd.DataFrame()
    for run_path in sorted(logs_dir.iterdir()):
        if not run_path.is_dir():
            continue

        run_name = run_path.name
        run_type = detect_run_type(run_name)
        # read only grasp logs
        if run_type != "grasp":
            continue

        # determine split flag
        split = "unseen" if "unseen" in run_name.lower() else "seen"

        # read used model checkpoint if present
        ckpt_path = run_path / "used_model_checkpoint.txt"
        used_model_checkpoint = None
        if ckpt_path.exists():
            try:
                used_model_checkpoint = ckpt_path.read_text().strip() or None
            except Exception:
                used_model_checkpoint = None

        results_dir = run_path / "results"
        if not results_dir.exists():
            continue

        for case_path in sorted(results_dir.glob("case*.txt")):
            parsed = parse_case_result(case_path)
            if not parsed:
                continue
            language_goal, metrics = parsed
            labels = METRIC_LABELS.get(len(metrics), [f"metric_{i}" for i in range(len(metrics))])
            row = {
                "run": run_name,
                "run_type": run_type,
                "split": split,
                "used_model_checkpoint": used_model_checkpoint,
                "case": case_path.stem,
                "language_goal": language_goal,
            }
            for label, value in zip(labels, metrics):
                row[label] = value
            rows.append(row)

    df = pd.DataFrame(rows)
    if df.empty:
        return df
    return df.sort_values(["run", "case"]).reset_index(drop=True)


In [4]:
test_results = load_test_results(logs_dir)
test_results.head()

,run,run_type,split,used_model_checkpoint,case,language_goal,avg_success,avg_step,avg_success_step,avg_reward
0,2025-11-02-22-47-09-test-grasp-a2,grasp,seen,/home/tressen-arms/Documents/experiments/Actio...,case0,grasp a round object,1.000000,2.200000,2.200000,0.811073
1,2025-11-02-22-47-09-test-grasp-a2,grasp,seen,/home/tressen-arms/Documents/experiments/Actio...,case1,get something to eat,1.000000,1.800000,1.800000,0.907900
2,2025-11-02-22-47-09-test-grasp-a2,grasp,seen,/home/tressen-arms/Documents/experiments/Actio...,case2,get something to hold other things,0.933333,1.666667,1.714286,0.712848
3,2025-11-02-22-47-09-test-grasp-a2,grasp,seen,/home/tressen-arms/Documents/experiments/Actio...,case3,I want a round object,1.000000,2.200000,2.200000,0.855614
4,2025-11-02-22-47-09-test-grasp-a2,grasp,seen,/home/tressen-arms/Documents/experiments/Actio...,case4,give me the cup,0.933333,3.600000,3.285714,0.553654


In [5]:
checkpoint_to_name = {
    "a2_pretrained": "pretrained_model",
    "2025-07-20-17-52-02-train-a2": "my_train_py38",
    "2025-11-02-22-57-28-train-a2": "my_train_py310",
    "2025-11-03-21-47-30-train-a2": "my_train_py310_v2",
}

In [6]:
test_results["checkpoint"] = test_results["used_model_checkpoint"].str.split("/").str[-3]
test_results["model_name"] = test_results["checkpoint"].map(checkpoint_to_name)
test_results["checkpoint"].unique()

array(['a2_pretrained', '2025-11-02-22-57-28-train-a2',
       '2025-07-20-17-52-02-train-a2', '2025-11-03-21-47-30-train-a2'],
      dtype=object)

In [7]:
metric_columns = [col for col in test_results.columns if col.startswith("avg_")]
summary = (
    test_results
    .groupby(["run", "run_type", "model_name", "split"])[metric_columns]
    .mean()
    .reset_index()
    .sort_values("model_name", ascending=False)
    .query("run_type == 'grasp'")
)
display(summary)


,run,run_type,model_name,split,avg_success,avg_step,avg_success_step,avg_reward
0,2025-11-02-22-47-09-test-grasp-a2,grasp,pretrained_model,seen,0.946667,2.080000,2.035556,0.728586
1,2025-11-02-23-07-20-test-grasp-a2-unseen,grasp,pretrained_model,unseen,0.933333,2.773333,2.515018,0.457278
5,2025-11-03-18-42-16-test-grasp-a2,grasp,my_train_py38,seen,0.906667,2.893333,2.624725,0.420037
7,2025-11-03-19-08-00-test-grasp-a2-unseen,grasp,my_train_py38,unseen,0.946667,2.800000,2.567912,0.515331
8,2025-11-03-23-04-59-test-grasp-a2,grasp,my_train_py310_v2,seen,0.920000,2.453333,2.436429,0.599707
9,2025-11-03-23-26-08-test-grasp-a2-unseen,grasp,my_train_py310_v2,unseen,0.800000,4.413333,3.690281,-0.279634
2,2025-11-03-16-20-54-test-grasp-a2,grasp,my_train_py310,seen,0.900000,2.773333,2.705833,0.495297
3,2025-11-03-16-44-04-test-grasp-a2-unseen,grasp,my_train_py310,unseen,0.746667,4.346667,3.612221,-0.326941
4,2025-11-03-18-20-06-test-grasp-a2,grasp,my_train_py310,seen,0.893333,2.733333,2.454634,0.473339
6,2025-11-03-18-43-54-test-grasp-a2-unseen,grasp,my_train_py310,unseen,0.760000,4.133333,3.224176,-0.197485
